In [2]:
import sys
print(sys.executable)

C:\Users\HP\GeoPulse--Hyper-Local-Retail-Mobility-Analytics\.venv\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("GeoPulse Environment Ready!")

GeoPulse Environment Ready!


In [4]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\HP\GeoPulse--Hyper-Local-Retail-Mobility-Analytics
['.git', '.gitignore', '.ipynb_checkpoints', '.venv', '02_gps_data_quality_eda.ipynb', '03_spatial_catchment_analysis.ipynb', '03_store_data_validation.ipynb', '04_store_cannibalization_analysis.ipynb', 'data', 'dbt', 'GeoPulse_Week1_GPS_Analysis.ipynb', 'notebooks', 'outputs', 'README.md', 'reports', 'sql', 'src', 'tests', 'Untitled.ipynb']


In [5]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.1.1
NumPy: 2.5.4


In [6]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "DeviceID": [f"D{i:03d}" for i in range(1, 101)],
    "Lat": np.random.uniform(28.60, 28.70, 100),
    "Long": np.random.uniform(77.10, 77.25, 100),
    "Timestamp": pd.date_range(
        start="2026-11-01",
        periods=100,
        freq="h"
    )
}

gps_data = pd.DataFrame(data)

gps_data.head()

,DeviceID,Lat,Long,Timestamp
0,D001,28.637454,77.104714,2026-11-01 00:00:00
1,D002,28.695071,77.195462,2026-11-01 01:00:00
2,D003,28.673199,77.147153,2026-11-01 02:00:00
3,D004,28.659866,77.176286,2026-11-01 03:00:00
4,D005,28.615602,77.236135,2026-11-01 04:00:00


In [7]:
gps_data.shape

(100, 4)

In [8]:
gps_data.columns

Index(['DeviceID', 'Lat', 'Long', 'Timestamp'], dtype='str')

In [9]:
gps_data.dtypes

DeviceID                str
Lat                 float64
Long                float64
Timestamp    datetime64[us]
dtype: object

In [10]:
gps_data.isnull().sum()

DeviceID     0
Lat          0
Long         0
Timestamp    0
dtype: int64

In [11]:
gps_data.duplicated().sum()

np.int64(0)

In [12]:
gps_data.describe()

,Lat,Long,Timestamp
count,100.000000,100.000000,100
mean,28.647018,77.174675,2026-11-03 01:30:00
min,28.600552,77.101043,2026-11-01 00:00:00
25%,28.619320,77.136301,2026-11-02 00:45:00
50%,28.646414,77.175844,2026-11-03 01:30:00
75%,28.673020,77.214928,2026-11-04 02:15:00
max,28.698689,77.247848,2026-11-05 03:00:00
std,0.029749,0.043967,NaN


In [13]:
gps_data.to_csv("geopulse_gps_data.csv", index=False)

print("GPS Dataset Saved Successfully!")

GPS Dataset Saved Successfully!


In [14]:
gps_check = pd.read_csv("geopulse_gps_data.csv")

print("Total Records:", len(gps_check))
print("Missing Values:")
print(gps_check.isnull().sum())

print("Duplicate Records:", gps_check.duplicated().sum())

Total Records: 100
Missing Values:
DeviceID     0
Lat          0
Long         0
Timestamp    0
dtype: int64
Duplicate Records: 0


In [15]:
print("Latitude Range:")
print(gps_check["Lat"].min(), gps_check["Lat"].max())

print("Longitude Range:")
print(gps_check["Long"].min(), gps_check["Long"].max())

Latitude Range:
28.600552211712362 28.698688693660053
Longitude Range:
77.10104281957967 77.24784756811658


In [16]:
stores = pd.DataFrame({
    "StoreID": ["S001", "S002", "S003", "S004", "S005"],
    "StoreName": [
        "Store A", "Store B", "Store C",
        "Store D", "Store E"
    ],
    "Lat": [28.6500, 28.6700, 28.6300, 28.6900, 28.6100],
    "Long": [77.1500, 77.1800, 77.2000, 77.1200, 77.2300]
})

stores

,StoreID,StoreName,Lat,Long
0,S001,Store A,28.65,77.15
1,S002,Store B,28.67,77.18
2,S003,Store C,28.63,77.20
3,S004,Store D,28.69,77.12
4,S005,Store E,28.61,77.23


In [17]:
stores.to_csv("geopulse_store_locations.csv", index=False)

print("Store Dataset Saved Successfully!")

Store Dataset Saved Successfully!


In [18]:
%pip install geopy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: C:\Users\HP\GeoPulse--Hyper-Local-Retail-Mobility-Analytics\.venv\Scripts\python.exe -m pip install --upgrade pip


In [19]:
from geopy.distance import geodesic

for i, store in stores.iterrows():

    count = 0

    for j, gps in gps_data.iterrows():

        distance = geodesic(
            (store["Lat"], store["Long"]),
            (gps["Lat"], gps["Long"])
        ).meters

        if distance <= 500:
            count += 1

    print(store["StoreName"], ":", count, "devices within 500 meters")

Store A : 1 devices within 500 meters
Store B : 0 devices within 500 meters
Store C : 1 devices within 500 meters
Store D : 1 devices within 500 meters
Store E : 1 devices within 500 meters


In [20]:
catchment_results = []

for i, store in stores.iterrows():
    count = 0

    for j, gps in gps_data.iterrows():
        distance = geodesic(
            (store["Lat"], store["Long"]),
            (gps["Lat"], gps["Long"])
        ).meters

        if distance <= 500:
            count += 1

    catchment_results.append({
        "StoreID": store["StoreID"],
        "StoreName": store["StoreName"],
        "Devices_Within_500m": count
    })

catchment_df = pd.DataFrame(catchment_results)

catchment_df

,StoreID,StoreName,Devices_Within_500m
0,S001,Store A,1
1,S002,Store B,0
2,S003,Store C,1
3,S004,Store D,1
4,S005,Store E,1


In [21]:
catchment_df.to_csv(
    "geopulse_catchment_analysis.csv",
    index=False
)

print("Catchment Analysis Saved Successfully!")

Catchment Analysis Saved Successfully!


In [22]:
# Check duplicate records
print("Duplicate records:", df.duplicated().sum())

# Check dataset information
print(df.info())

# Check latitude range
print("Latitude range:", df['Lat'].min(), df['Lat'].max())

# Check longitude range
print("Longitude range:", df['Long'].min(), df['Long'].max())

NameError: name 'df' is not defined

In [23]:
import pandas as pd

print(df.head())

NameError: name 'df' is not defined

In [24]:
%whos

Variable               Type         Data/Info
---------------------------------------------
catchment_df           DataFrame    Shape: (5, 3)
catchment_results      list         n=5
count                  int          1
data                   dict         n=4
dataframe_columns      function     <function dataframe_colum<...>ns at 0x000002092A063740>
dataframe_hash         function     <function dataframe_hash at 0x000002091822B950>
distance               float        1276.2526159430793
dtypes_str             function     <function dtypes_str at 0x000002091822BAB0>
geodesic               type         <class 'geopy.distance.geodesic'>
get_dataframes         function     <function get_dataframes at 0x00000209285CC300>
getpass                module       <module 'getpass' from 'C<...>hon314\\Lib\\getpass.py'>
gps                    Series       Shape: (4,)
gps_check              DataFrame    Shape: (100, 4)
gps_data               DataFrame    Shape: (100, 4)
hashlib                module  

In [25]:
print(gps_data.head())

  DeviceID        Lat       Long           Timestamp
0     D001  28.637454  77.104714 2026-11-01 00:00:00
1     D002  28.695071  77.195462 2026-11-01 01:00:00
2     D003  28.673199  77.147153 2026-11-01 02:00:00
3     D004  28.659866  77.176286 2026-11-01 03:00:00
4     D005  28.615602  77.236135 2026-11-01 04:00:00


In [26]:
# 1. Check duplicate records
print("Duplicate records:", gps_data.duplicated().sum())

# 2. Check missing values
print("\nMissing Values:")
print(gps_data.isnull().sum())

# 3. Check latitude range
print("\nLatitude Range:")
print(gps_data['Lat'].min(), gps_data['Lat'].max())

# 4. Check longitude range
print("\nLongitude Range:")
print(gps_data['Long'].min(), gps_data['Long'].max())

# 5. Check timestamp datatype
print("\nTimestamp Datatype:")
print(gps_data['Timestamp'].dtype)

Duplicate records: 0

Missing Values:
DeviceID     0
Lat          0
Long         0
Timestamp    0
dtype: int64

Latitude Range:
28.600552211712362 28.698688693660053

Longitude Range:
77.10104281957967 77.24784756811658

Timestamp Datatype:
datetime64[us]


In [27]:
# Validate latitude and longitude
gps_data['Valid_Coordinates'] = (
    gps_data['Lat'].between(-90, 90) &
    gps_data['Long'].between(-180, 180)
)

print("Valid coordinates:",
      gps_data['Valid_Coordinates'].sum())

print("Invalid coordinates:",
      (~gps_data['Valid_Coordinates']).sum())

Valid coordinates: 100
Invalid coordinates: 0


In [28]:
print("Earliest GPS ping:", gps_data['Timestamp'].min())
print("Latest GPS ping:", gps_data['Timestamp'].max())
print("Total GPS records:", len(gps_data))

Earliest GPS ping: 2026-11-01 00:00:00
Latest GPS ping: 2026-11-05 03:00:00
Total GPS records: 100


In [29]:
# Create a clean GPS dataset
gps_clean = gps_data.copy()

# Convert timestamp into datetime
gps_clean['Timestamp'] = pd.to_datetime(
    gps_clean['Timestamp']
)

# Sort records by timestamp
gps_clean = gps_clean.sort_values(
    by='Timestamp'
).reset_index(drop=True)

# Display cleaned dataset
print(gps_clean.head())

print("\nFinal Dataset Shape:", gps_clean.shape)
print("\nData Types:")
print(gps_clean.dtypes)

  DeviceID        Lat       Long           Timestamp  Valid_Coordinates
0     D001  28.637454  77.104714 2026-11-01 00:00:00               True
1     D002  28.695071  77.195462 2026-11-01 01:00:00               True
2     D003  28.673199  77.147153 2026-11-01 02:00:00               True
3     D004  28.659866  77.176286 2026-11-01 03:00:00               True
4     D005  28.615602  77.236135 2026-11-01 04:00:00               True

Final Dataset Shape: (100, 5)

Data Types:
DeviceID                        str
Lat                         float64
Long                        float64
Timestamp            datetime64[us]
Valid_Coordinates              bool
dtype: object


In [30]:
gps_clean.to_csv(
    'gps_data_clean.csv',
    index=False
)

print("Clean GPS dataset saved successfully!")

Clean GPS dataset saved successfully!


In [31]:
gps_clean.to_csv('gps_data_clean.csv', index=False)

print("GPS data saved successfully!")

GPS data saved successfully!


In [32]:
import os

print("File exists:", os.path.exists('gps_data_clean.csv'))
print("File size:", os.path.getsize('gps_data_clean.csv'), "bytes")

File exists: True
File size: 6784 bytes
